   ![alt text](https://www.mbari.org/wp-content/uploads/2014/11/logo-mbari-3b.png "MBARI")

  <div align="left">Copyright (c) 2021, MBARI</div>

  * Distributed under the terms of the GPL License
  * Maintainer: dcline@mbari.org
  * Authors: Danelle Cline dcline@mbari.org, woestreich@stanford.edu


## Applying Machine Learning to classify blue whale D calls

---


## Install dependencies
First, let's install dependencies and include all packages used in this notebook. This only needs to be done once for the duration of this notebook.

In [1]:
# !pip install h5py==3.4.0
!pip install oceansoundscape==1.0.1  --quiet
!pip install boto3  --quiet
!pip install pandas==1.1.0  --quiet
!pip install tensorflow==2.4.1 --quiet
# !pip install numpy==1.20 
# !pip install pandas==1.3.2 
#!pip check

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.4.1 requires h5py~=2.10.0, but you have h5py 3.4.0 which is incompatible.
tensorflow 2.4.1 requires numpy~=1.19.2, but you have numpy 1.20.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
oceansoundscape 1.0.1 requires pandas==1.3.2, but you have pandas 1.1.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
oceansoundscape 1.0.1 requires h5py==3.4.0, but you have h5py 2.10.0 which is incompatible.
oceansoundscape 1.0.1 requires numpy==1.20, but you have numpy 1.19.5 which is incompatible.
oceansoundscape 1.

In [2]:
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import glob
import os
import datetime as dt
from PIL import Image
import oceansoundscape
from oceansoundscape.spectrogram import conf, denoise, utils
from oceansoundscape.raven import BLEDParser
from pathlib import Path
import soundfile as sf
import json
import numpy as np
import tensorflow as tf


### Download 2 kHz files for each BLED file

For each audited file, download the associated 2 kHz file from the pacific-sound-2khz bucket.


In [3]:
session = boto3.Session()
s3_session = session.resource('s3', config=Config(signature_version=UNSIGNED))

for filename in glob.glob('*.txt'): 
  wav_filename = filename.split('.audit')[0]  
  mars_dt = dt.datetime.strptime(wav_filename, "MARS-%Y%m%dT%H%M%SZ-2kHz.wav")
  key = f'{mars_dt.year:04}/{mars_dt.month:02}/{wav_filename}'
  print(f'Downloading {wav_filename}')
  s3_session.Bucket('pacific-sound-2khz').download_file(key, wav_filename)
  print('Done')

Done


### Download the model
 
First, download that model then uncompress it.  The model should be version 1 and exist in a subdirectory simply called "1".


There are two models that you can use: a resnet50 model, and an efficientnetb0 model.  

In [11]:
bucket = 'pacific-sound-models'
model_1 = 'bluewhale-d-resnet50-2021-09-24-03-42-22-608.tar.gz'
# model_2 = 'bluewhale-d-efnetb0-2021-08-31-14-37-02-268.tar.gz'
# model_3 = 'bluewhale-d-resnet50-2021-09-25-20-53-09-072.tar.gz'
# model_4 = 'bluewhale-d-resnet50-2021-09-30-01-03-04-165.tar.gz'

target_model = model_1 

session = boto3.Session()

s3_session = session.resource('s3', config=Config(signature_version=UNSIGNED))

print(f'Downloading {target_model}...')
s3_session.Bucket(bucket).download_file(target_model, target_model)

print(f'Uncompressing {target_model}')
!tar -xf {target_model}
print(f'Done')

Uncompressing bluewhale-d-resnet50-2021-09-24-03-42-22-608.tar.gz
Done


In [12]:
os.listdir("1")

['config.json', 'variables', 'saved_model.pb', 'assets']

### Load the model

In [13]:
model = tf.keras.models.load_model("1")
# ignore the wanings about importing with ops with unsaved custom gradients - gradients are only used during training

### Show the model architecture and grab the normalization parameters

Here, we can see the last layer only this model only outputs a 2 wide vector - this is for either blue whale D false or blue whale D true calls.   Class labels are defined in the config.json with the model as are the normalization parameters.

In [14]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
resnet50 (Functional)        (None, 7, 7, 2048)        23587712  
_________________________________________________________________
global_average_pooling2d (Gl (None, 2048)              0         
_________________________________________________________________
dense (Dense)                (None, 2)                 4098      
Total params: 23,591,810
Trainable params: 23,538,690
Non-trainable params: 53,120
_________________________________________________________________


In [15]:
config = json.load(open('1/config.json'))
image_mean = np.asarray(config["image_mean"])
image_std = np.asarray(config["image_std"])
print(f"Labels {config['classes']}")
print(f"Training image mean: {image_mean}")
print(f"Training image std: {image_std}")

Labels ['bdf', 'bdt']
Training image mean: [0.16522324 0.63128757 0.7104407 ]
Training image std: [0.04085101 0.02235668 0.01962656]


## Parse each audited file and process


Now, run all the spectrogram images and store back into the Raven table

In [16]:
def classify(wav_path:Path, bled_path:Path, model_name: str):
    """
     Runs classification on a BLED file
    :param wav_file:  2 khz wav file path associated with the BLED file
    :param bled_path:  the Band-Limited-Energy-Detection file path
    :return:
    """
    # an optimum configuration for the call spectrogram generation is defined in the oceanscoundscape package based
    # on extensive hyperparameter sweeps. Let's grab those here
    blue_conf = conf.CONF_DICT['blueD']

    # read the wav file
    xc, sample_rate = sf.read(wav_path.name)
     
    # grab the spectrogram parameters
    call_width = int(blue_conf['duration_secs'] * sample_rate)
    num_fft = blue_conf['num_fft']
    hop_length = round(num_fft * (1 - conf.OVERLAP))
    batch_size = 1

    # parse the detections, but this time let's use the entire wav file to allow for padding during denoising
    parser = BLEDParser(bled_path, blue_conf, len(xc), sample_rate)

    # generate optimized spectrograms
    print(f'Denoising {bled_path} num detections {parser.num_detections}')

    # denoise them and store the results in a temporary hdf file cache
    cache_path_hdf = Path(f'{bled_path.name}.hdf')
    cache = denoise.PCENSmooth(cache_path_hdf, parser, blue_conf, xc, sample_rate)

    df = parser.data
    bdt = []
    bdf = []
    num_bdt = 0
    num_bdt_c = 0
    for row, item in sorted(parser.data.iterrows()):
        try:
            if item.Classification == 'bdt':  
                print(f'Running model on {item.image_filename}')
                
                image_path = Path(item.image_filename)
                data = cache.get_data(item.Selection)
                
                start = int(call_width / hop_length)
                end = int(2*call_width/ hop_length)

                # subset the call, leaving off the padding for PCEN
                subset_data = data[:, start:end]

                # save to a denoised color image
                utils.ImageUtils.colorizeDenoise(subset_data, image_path)

                # open the image and normalize
                image = Image.open(item.image_filename).convert('RGB')
                image_float = np.asarray(image).astype('float32')
                image_float = image_float / 255.
                image_float = (image_float - image_mean) / ( image_std + 1.e-9)
        
                # put into a 4-D stack of batch size 1 and run the model
                image = np.concatenate([image_float[np.newaxis, :, :]] * batch_size)
                tensor_out = model(image)

                # the model outputs a tensor, so let's convert to a numpy array here
                predictions = tensor_out.numpy()[0]

                # pull out the prediction
                bdf.append(predictions[0])
                bdt.append(predictions[1])

                # clean-up
#           image_path.unlink()

                print(f'{item.Classification} {predictions[0]:0.04} {predictions[1]:0.04}')
            
                num_bdt += 1
                if predictions[1] < 0.7:
                    shutil.move(item.image_filename, f'incorrect/{int(predictions[1]*100)}_{item.image_filename}')
                else:
                    num_bdt_c += 1
                    shutil.move(item.image_filename, f'correct/{int(predictions[1]*100)}_{item.image_filename}')
            
                if item.Classification == 'bdf' and predictions[1] > 0.5:  
                    print(f'{item.Classification} {predictions[0]:0.04} {predictions[1]:0.04}')
#             if row > 20:
#               break;

        except Exception as ex:
            # place holders for failed predictions - this can only happen when
            # the bled detection crosses over the boundary at the beginning 
            # or the ending of the file causing and 'out of bounds' error
            bdf.append(-1.)
            bdt.append(-1.)
            print(ex)
            continue
            
    print(f'num_bdt {num_bdt} num_bdt_correct {num_bdt_c}')

    # store the predictions back into the data frame and save to a csv
#     df['bdf'] = bdf
#     df['bdt'] = bdt
#     df.to_csv(f'{bled_path.stem}_{model_name}.csv')

    # remove the cache file
#     cache_path_hdf.unlink()

In [17]:
import shutil
for bled_file in glob.glob('*.txt'): 
  wav_file = bled_file.split('.audit')[0]
  classify(Path(wav_file), Path(bled_file), target_model)

Reading MARS-20170626T000000Z-2kHz.wav.audit.txt {'low_freq': 20, 'high_freq': 75, 'duration_secs': 7, 'blur_axis': '', 'num_fft': 1024, 'center': True, 'padding_secs': 2, 'num_mels': 30}
Found 1107 detections in MARS-20170626T000000Z-2kHz.wav.audit.txt
Denoising MARS-20170626T000000Z-2kHz.wav.audit.txt num detections 1107
Skipping 0 call out of bounds
Running model on 20170626T003142_bdt.3804550.3806743.sel.10.ch01.spectrogram.jpg
bdt 0.9239 0.07609
Running model on 20170626T003318_bdt.3997381.4001461.sel.12.ch01.spectrogram.jpg
bdt 0.9352 0.06478
Running model on 20170626T003527_bdt.4255390.4257787.sel.14.ch01.spectrogram.jpg
bdt 0.9405 0.05953
Running model on 20170626T003659_bdt.4438888.4441744.sel.16.ch01.spectrogram.jpg
bdt 0.9417 0.05826
Running model on 20170626T004006_bdt.4813942.4816747.sel.18.ch01.spectrogram.jpg
bdt 0.9241 0.07591
Running model on 20170626T005722_bdt.6884644.6886990.sel.27.ch01.spectrogram.jpg
bdt 0.7398 0.2602
Running model on 20170626T005748_bdt.6937480.6

bdt 0.5047 0.4953
Running model on 20170626T165350_bdt.121661827.121664785.sel.640.ch01.spectrogram.jpg
bdt 0.2734 0.7266
Running model on 20170626T165429_bdt.121738174.121741591.sel.641.ch01.spectrogram.jpg
bdt 0.111 0.889
Running model on 20170626T165508_bdt.121817173.121819315.sel.642.ch01.spectrogram.jpg
bdt 0.5656 0.4344
Running model on 20170626T182449_bdt.132579142.132581590.sel.661.ch01.spectrogram.jpg
bdt 0.9283 0.07167
Running model on 20170626T183528_bdt.133856386.133858477.sel.667.ch01.spectrogram.jpg
bdt 0.9356 0.06443
Running model on 20170626T185522_bdt.136244257.136249204.sel.677.ch01.spectrogram.jpg
bdt 0.9448 0.05524
Running model on 20170626T185810_bdt.136581826.136583917.sel.680.ch01.spectrogram.jpg
bdt 0.93 0.06995
Running model on 20170626T185840_bdt.136640323.136644250.sel.681.ch01.spectrogram.jpg
bdt 0.9435 0.05646
Running model on 20170626T185843_bdt.136647106.136650268.sel.682.ch01.spectrogram.jpg
bdt 0.9374 0.06265
Running model on 20170626T190119_bdt.1369589

bdt 0.1042 0.8958
Running model on 20170626T204402_bdt.149284651.149288833.sel.817.ch01.spectrogram.jpg
bdt 0.9127 0.08733
Running model on 20170626T204433_bdt.149347738.149351512.sel.818.ch01.spectrogram.jpg
bdt 0.9456 0.0544
Running model on 20170626T204525_bdt.149450146.149457082.sel.819.ch01.spectrogram.jpg
bdt 0.09905 0.9009
Running model on 20170626T204720_bdt.149680564.149687449.sel.820.ch01.spectrogram.jpg
bdt 0.3579 0.6421
Running model on 20170626T204826_bdt.149813113.149820304.sel.821.ch01.spectrogram.jpg
bdt 0.08913 0.9109
Running model on 20170626T205404_bdt.150488149.150494524.sel.828.ch01.spectrogram.jpg
bdt 0.2637 0.7363
Running model on 20170626T205450_bdt.150581173.150584845.sel.829.ch01.spectrogram.jpg
bdt 0.9256 0.07437
Running model on 20170626T205809_bdt.150979228.150988204.sel.832.ch01.spectrogram.jpg
bdt 0.07315 0.9269
Running model on 20170626T205851_bdt.151063429.151069957.sel.835.ch01.spectrogram.jpg
bdt 0.04833 0.9517
Running model on 20170626T210836_bdt.152

bdt 0.03716 0.9628
Running model on 20170626T232930_bdt.169140736.169142980.sel.1082.ch01.spectrogram.jpg
bdt 0.9444 0.05564
Running model on 20170626T233041_bdt.169283434.169287157.sel.1084.ch01.spectrogram.jpg
bdt 0.9364 0.06363
Running model on 20170626T233117_bdt.169354477.169361260.sel.1085.ch01.spectrogram.jpg
bdt 0.05558 0.9444
Running model on 20170626T233125_bdt.169371460.169376611.sel.1086.ch01.spectrogram.jpg
bdt 0.07927 0.9207
Running model on 20170626T233135_bdt.169390126.169392982.sel.1087.ch01.spectrogram.jpg
bdt 0.4895 0.5105
Running model on 20170626T233256_bdt.169553632.169558324.sel.1089.ch01.spectrogram.jpg
bdt 0.9494 0.05055
Running model on 20170626T233420_bdt.169721983.169726369.sel.1091.ch01.spectrogram.jpg
bdt 0.1348 0.8652
Running model on 20170626T233545_bdt.169890232.169896505.sel.1093.ch01.spectrogram.jpg
bdt 0.07068 0.9293
Running model on 20170626T233612_bdt.169945057.169948576.sel.1094.ch01.spectrogram.jpg
bdt 0.7239 0.2761
Running model on 20170626T2336

In [ ]:
num_bdt 252 num_bdt_correct 88  model_1
num_bdt 252 num_bdt_correct 59  model_3 with dropout
num_bdt 252 num_bdt_correct 64  model_4